## Simple ChemBL parser for creating random datasets

In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from chembl_webresource_client.new_client import new_client

/home/mark/miniforge3/envs/ml1/lib/python3.14/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


In [3]:
molecule_api = new_client.molecule

N = 10_000  # нужный размер сабсета

# Берём drug-like молекулы с MW 200–600, есть SMILES
molecules = molecule_api.filter(
    molecule_properties__mw_freebase__gte=200,
    molecule_properties__mw_freebase__lte=600,
    molecule_properties__alogp__gte=-2,
    molecule_structures__isnull=False,
).only("molecule_chembl_id", "molecule_structures")

# Собираем SMILES
records = []
for mol in tqdm(molecules[:N]):
    try:
        smi = mol["molecule_structures"]["canonical_smiles"]
        if smi:
            records.append({
                "chembl_id": mol["molecule_chembl_id"],
                "smiles": smi
            })
    except (KeyError, TypeError):
        continue

df = pd.DataFrame(records).drop_duplicates("smiles").reset_index(drop=True)
print(f"Собрано молекул: {len(df)}")


  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [16:42<00:00,  9.98it/s]

Собрано молекул: 10000


In [4]:
desc_names = [x[0] for x in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return calc.CalcDescriptors(mol)

tqdm.pandas()
df["descriptors"] = df["smiles"].progress_apply(compute_descriptors)

# Убираем невалидные молекулы
df = df[df["descriptors"].notna()].reset_index(drop=True)

# Разворачиваем в колонки
desc_df = pd.DataFrame(df["descriptors"].tolist(), columns=desc_names)
result = pd.concat([df[["chembl_id", "smiles"]], desc_df], axis=1)


100%|██████████| 10000/10000 [01:08<00:00, 145.46it/s]


In [5]:
your_features = [
    "Smiles", "pIC50", "MaxAbsEStateIndex", "MinAbsEStateIndex", "MinEStateIndex",
    "qed", "SPS", "MolWt", "MaxPartialCharge", "MinPartialCharge",
    "FpDensityMorgan1", "BCUT2D_MWHI", "BCUT2D_MWLOW", "BCUT2D_CHGHI",
    "BCUT2D_CHGLO", "BCUT2D_LOGPHI", "BCUT2D_LOGPLOW", "BCUT2D_MRHI",
    "BCUT2D_MRLOW", "AvgIpc", "BalabanJ", "BertzCT", "HallKierAlpha",
    "PEOE_VSA1", "PEOE_VSA10", "PEOE_VSA11", "PEOE_VSA12", "PEOE_VSA13",
    "PEOE_VSA14", "PEOE_VSA2", "PEOE_VSA3", "PEOE_VSA4", "PEOE_VSA5",
    "PEOE_VSA6", "PEOE_VSA7", "PEOE_VSA8", "PEOE_VSA9", "SMR_VSA1",
    "SMR_VSA10", "SMR_VSA2", "SMR_VSA3", "SMR_VSA4", "SMR_VSA5",
    "SMR_VSA6", "SMR_VSA7", "SMR_VSA9", "SlogP_VSA1", "SlogP_VSA10",
    "SlogP_VSA11", "SlogP_VSA12", "SlogP_VSA2", "SlogP_VSA3", "SlogP_VSA4",
    "SlogP_VSA5", "SlogP_VSA7", "SlogP_VSA8", "TPSA", "EState_VSA1",
    "EState_VSA10", "EState_VSA11", "EState_VSA2", "EState_VSA3", "EState_VSA4",
    "EState_VSA5", "EState_VSA6", "EState_VSA7", "EState_VSA8", "EState_VSA9",
    "VSA_EState1", "VSA_EState2", "VSA_EState3", "VSA_EState4", "VSA_EState5",
    "VSA_EState6", "VSA_EState7", "VSA_EState8", "VSA_EState9", "FractionCSP3",
    "NHOHCount", "NumAliphaticCarbocycles", "NumAliphaticHeterocycles",
    "NumAliphaticRings", "NumAmideBonds", "NumAromaticCarbocycles",
    "NumAromaticHeterocycles", "NumAromaticRings", "NumAtomStereoCenters",
    "NumBridgeheadAtoms", "NumHAcceptors", "NumHeterocycles",
    "NumSaturatedCarbocycles", "NumSaturatedHeterocycles", "NumSaturatedRings",
    "NumSpiroAtoms", "NumUnspecifiedAtomStereoCenters", "RingCount", "MolLogP",
    "fr_Al_COO", "fr_Al_OH", "fr_Al_OH_noTert", "fr_ArN", "fr_Ar_COO",
    "fr_Ar_N", "fr_Ar_NH", "fr_Ar_OH", "fr_COO", "fr_C_S", "fr_HOCCN",
    "fr_Imine", "fr_NH0", "fr_NH1", "fr_NH2", "fr_N_O", "fr_Ndealkylation1",
    "fr_Ndealkylation2", "fr_SH", "fr_aldehyde", "fr_alkyl_carbamate",
    "fr_alkyl_halide", "fr_allylic_oxid", "fr_amidine", "fr_aniline",
    "fr_aryl_methyl", "fr_azide", "fr_azo", "fr_benzodiazepine", "fr_bicyclic",
    "fr_dihydropyridine", "fr_epoxide", "fr_ester", "fr_ether", "fr_furan",
    "fr_guanido", "fr_halogen", "fr_hdrzine", "fr_hdrzone", "fr_imidazole",
    "fr_imide", "fr_ketone", "fr_lactam", "fr_lactone", "fr_methoxy",
    "fr_morpholine", "fr_nitro", "fr_nitro_arom_nonortho", "fr_nitroso",
    "fr_oxazole", "fr_oxime", "fr_para_hydroxylation", "fr_phenol",
    "fr_phos_acid", "fr_piperdine", "fr_piperzine", "fr_priamide",
    "fr_prisulfonamd", "fr_pyridine", "fr_quatN", "fr_sulfide", "fr_sulfonamd",
    "fr_sulfone", "fr_term_acetylene", "fr_tetrazole", "fr_thiazole",
    "fr_thiophene", "fr_unbrch_alkane", "fr_urea"
]




cols_to_keep = ["chembl_id", "smiles"] + [
    c for c in your_features if c in result.columns
]
result = result[cols_to_keep]

result.to_csv("chembl_random_subset.csv", index=False)
print(f"Сохранено: {result.shape}")

Сохранено: (10000, 166)
